# Archetypes Exploration

Visual exploration of archetype features to get a qualitative impression of how well the archetype labels and queries work.
Covers the four core graph metrics computed in the archetypes domain: PageRank, ArticleRank, Betweenness, and Local Clustering Coefficient.
Includes Pareto-frontier analysis to identify code units with the most extreme trade-offs for each archetype.

<br>

### Archetypes

- **Authority**: High PageRank with a disproportionately large gap to ArticleRank. The code unit is widely referenced but does not reference others proportionally. Suggests utility libraries or framework entry points.
- **Hub**: High degree (many connections) but low local clustering coefficient (neighbors are not well connected to each other). Suggests God classes or overly broad utilities that violate modularity.
- **Bottleneck**: High betweenness centrality. The code unit lies on many shortest dependency paths between other nodes, acting as an architectural chokepoint.

### References
- [jqassistant](https://jqassistant.org)
- [Neo4j Python Driver](https://neo4j.com/docs/api/python-driver/current)

In [ ]:
import os
import typing

from IPython.display import display
import pandas as pd
import numpy as np
import matplotlib.pyplot as plot
import seaborn

In [ ]:
%%html
<style>
/* Smaller font size for wide DataFrame tables in notebook output. */
.dataframe th {
    font-size: 8px;
}
.dataframe td {
    font-size: 8px;
}
</style>

In [ ]:
MAIN_COLOR_MAP = 'viridis'

plot_annotation_style: dict = {
    'textcoords': 'offset points',
    'arrowprops': dict(arrowstyle='->', color='black', alpha=0.3),
    'fontsize': 6,
    'backgroundcolor': 'white',
    'bbox': dict(
        boxstyle='round,pad=0.4',
        edgecolor='silver',
        facecolor='whitesmoke',
        alpha=1,
    ),
}

In [ ]:
from sys import version as python_version
print(f"Python version: {python_version}")

from numpy import __version__ as numpy_version
print(f"numpy version: {numpy_version}")

from pandas import __version__ as pandas_version
print(f"pandas version: {pandas_version}")

from matplotlib import __version__ as matplotlib_version
print(f"matplotlib version: {matplotlib_version}")

from seaborn import __version__ as seaborn_version  # type: ignore
print(f"seaborn version: {seaborn_version}")

from neo4j import __version__ as neo4j_version
print(f"neo4j version: {neo4j_version}")

In [ ]:
# NEO4J_INITIAL_PASSWORD must be set in the shell before starting the notebook.
from neo4j import GraphDatabase

driver = GraphDatabase.driver(
    uri="bolt://localhost:7687",
    auth=("neo4j", os.environ.get("NEO4J_INITIAL_PASSWORD")),
)
driver.verify_connectivity()

In [ ]:
def query_cypher_to_data_frame(
    query: typing.LiteralString,
    parameters: dict[str, typing.Any] | None = None,
) -> pd.DataFrame:
    """Execute a parameterized Cypher query and return the results as a DataFrame."""
    records, _, keys = driver.execute_query(query, parameters_=parameters)
    return pd.DataFrame([record.values() for record in records], columns=keys)


def query_archetype_data(
    node_label: typing.Literal[
        "Package",
        "Type",
        "SemanticCodeIndexModule",
        "SemanticCodeIndexInternalType",
    ],
) -> pd.DataFrame:
    """Query all nodes with the given label that have archetype feature properties populated.

    Returns a DataFrame with one row per code unit and columns for all archetype features,
    plus boolean flags indicating which archetype labels have been applied.
    Supports Java (Package, Type) and SCIP (SemanticCodeIndexModule, SemanticCodeIndexInternalType) nodes.
    """
    query: typing.LiteralString = """
        MATCH (codeUnit)
        WHERE $projection_node_label IN labels(codeUnit)
          AND codeUnit.centralityPageRank                        IS NOT NULL
          AND codeUnit.centralityArticleRank                     IS NOT NULL
          AND codeUnit.centralityPageRankToArticleRankDifference IS NOT NULL
          AND codeUnit.centralityBetweenness                     IS NOT NULL
          AND codeUnit.communityLocalClusteringCoefficient       IS NOT NULL
          AND codeUnit.incomingDependencies                      IS NOT NULL
          AND codeUnit.outgoingDependencies                      IS NOT NULL
        WITH *
            ,coalesce(codeUnit.fqn, codeUnit.globalFqn, codeUnit.fileName, codeUnit.signature, codeUnit.name) AS codeUnitName
            ,coalesce(codeUnit.projectName, '')                                  AS projectName
        RETURN DISTINCT
             codeUnitName
            ,codeUnit.name                                                       AS shortCodeUnitName
            ,projectName
            ,codeUnit.incomingDependencies                                       AS incomingDependencies
            ,codeUnit.outgoingDependencies                                       AS outgoingDependencies
            ,codeUnit.incomingDependencies + codeUnit.outgoingDependencies       AS degree
            ,codeUnit.centralityPageRank                                         AS pageRank
            ,codeUnit.centralityArticleRank                                      AS articleRank
            ,codeUnit.centralityPageRankToArticleRankDifference                  AS pageToArticleRankDifference
            ,codeUnit.centralityBetweenness                                      AS betweenness
            ,codeUnit.communityLocalClusteringCoefficient                        AS localClusteringCoefficient
            ,1.0 - codeUnit.communityLocalClusteringCoefficient                  AS inverseLocalClusteringCoefficient
            ,CASE WHEN 'Mark4TopArchetypeHub'        IN labels(codeUnit) THEN true ELSE false END AS isHubArchetype
            ,CASE WHEN 'Mark4TopArchetypeAuthority'  IN labels(codeUnit) THEN true ELSE false END AS isAuthorityArchetype
            ,CASE WHEN 'Mark4TopArchetypeBottleneck' IN labels(codeUnit) THEN true ELSE false END AS isBottleneckArchetype
    """
    return query_cypher_to_data_frame(query, {"projection_node_label": node_label})

In [ ]:
def truncate(text: str, max_length: int = 26) -> str:
    """Truncate text to max_length, replacing the tail with dots when the limit is exceeded."""
    if len(text) <= max_length:
        return text
    return text[:max_length - 3] + "..."


def annotate_most_extreme_points(
    data: pd.DataFrame,
    name_column: str,
    x_column: str,
    y_column: str,
    sort_column: str,
    ascending: bool = False,
    top_n: int = 10,
    text_offset_x: int = 5,
) -> None:
    """Annotate the top_n most extreme data points in a scatter plot.

    Args:
        data: The DataFrame to select points from.
        name_column: Column containing display labels for the annotations.
        x_column: Column for the x position of the annotated point.
        y_column: Column for the y position of the annotated point.
        sort_column: Column used to rank extremeness.
        ascending: If True, annotate the smallest values (e.g. lowest clustering coefficient).
        top_n: Number of points to annotate.
        text_offset_x: Horizontal text offset in points from the data point.
    """
    if data.empty:
        return
    top_points = (
        data.nsmallest(top_n, sort_column) if ascending
        else data.nlargest(top_n, sort_column)
    ).reset_index(drop=True)
    for rank, (_, row) in enumerate(top_points.iterrows(), start=1):
        y_offset = (rank % 5) * 10
        plot.annotate(
            text=f"#{rank}: {truncate(str(row[name_column]))} ({row[sort_column]:.3f})",
            xy=(row[x_column], row[y_column]),
            xytext=(text_offset_x, 5 + y_offset),
            **plot_annotation_style,
        )


def plot_standard_deviation_lines(
    color: typing.LiteralString,
    mean: float,
    standard_deviation: float,
    standard_deviation_factor: int = 0,
) -> None:
    """Plot a vertical dashed line at mean + factor * standard_deviation."""
    value = mean + standard_deviation_factor * standard_deviation
    if standard_deviation_factor == 0:
        label = f"Mean: {value:.2f}"
    else:
        label = f"Mean + {standard_deviation_factor} x Std Dev: {value:.2f}"
    plot.axvline(value, color=color, linestyle='dashed', linewidth=1, label=label)
    if standard_deviation_factor != 0:
        plot.axvline(
            mean - standard_deviation_factor * standard_deviation,
            color=color,
            linestyle='dashed',
            linewidth=1,
        )
    plot.legend()


def plot_percentile_lines(
    feature_values: pd.Series,
    percentile_in_percent: int,
    color: typing.LiteralString = 'purple',
) -> None:
    """Plot a vertical dotted line at the given percentile of feature_values."""
    value = float(np.percentile(feature_values.dropna(), percentile_in_percent))
    plot.axvline(value, color=color, linestyle='dotted', linewidth=1.2,
                 label=f"Percentile {percentile_in_percent}: {value:.2f}")
    plot.legend()


def compute_tukey_fence(
    values: pd.Series,
    fence_multiplier: float,
    upper: bool = True,
) -> float:
    """Compute a Tukey fence threshold.

    Args:
        values: The data series to compute the fence for.
        fence_multiplier: Multiplier applied to the interquartile range (e.g. 1.0 or 1.5).
        upper: If True, return Q3 + multiplier*IQR; if False, return Q1 - multiplier*IQR.

    Returns:
        The fence value as a float.
    """
    clean_values = values.dropna()
    first_quartile = float(clean_values.quantile(0.25))
    third_quartile = float(clean_values.quantile(0.75))
    interquartile_range = third_quartile - first_quartile
    if upper:
        return third_quartile + fence_multiplier * interquartile_range
    return first_quartile - fence_multiplier * interquartile_range


def plot_tukey_fence_upper_line(
    feature_values: pd.Series,
    fence_multiplier: float,
    color: typing.LiteralString,
) -> None:
    """Plot a vertical dash-dot line at the upper Tukey fence (Q3 + fence_multiplier * IQR)."""
    upper_fence = compute_tukey_fence(feature_values, fence_multiplier, upper=True)
    plot.axvline(
        upper_fence,
        color=color,
        linestyle='dashdot',
        linewidth=1.2,
        label=f"Tukey upper fence ({fence_multiplier}xIQR): {upper_fence:.2f}",
    )
    plot.legend()


In [ ]:
def plot_feature_distribution(
    feature_values: pd.Series,
    feature_name: str,
    short_names: pd.Series,
    title_prefix: str,
    log_y_axis: bool = False,
    standard_deviation_lines: bool = True,
    number_of_outliers_to_annotate: int = 4,
) -> None:
    """Plot a histogram of feature_values with optional standard deviation reference lines and outlier annotations.

    Args:
        feature_values: The values to plot.
        feature_name: Label for the x-axis and title.
        short_names: Short code unit names aligned with feature_values by index (used for annotation).
        title_prefix: Prefix prepended to the plot title (e.g. the node type).
        log_y_axis: Use a logarithmic y-axis scale.
        standard_deviation_lines: Overlay mean and ±1, ±2 standard deviation lines.
        number_of_outliers_to_annotate: Number of extreme values to label on each end.
    """
    if feature_values.empty:
        print("No data available to plot.")
        return

    mean = feature_values.mean()
    standard_deviation = feature_values.std()

    plot.figure(figsize=(10, 6))
    plot.hist(feature_values, bins=40, color='steelblue', alpha=0.7, edgecolor='black')
    plot.title(f"{title_prefix} Distribution of {feature_name}", pad=20)
    plot.xlabel(feature_name)
    plot.ylabel('Frequency')
    plot.xlim(left=feature_values.min(), right=feature_values.max())
    if log_y_axis:
        plot.yscale('log')
    plot.grid(True)
    plot.tight_layout()

    if standard_deviation_lines:
        plot_standard_deviation_lines('red', mean, standard_deviation, standard_deviation_factor=0)
        plot_standard_deviation_lines('orange', mean, standard_deviation, standard_deviation_factor=1)
        plot_standard_deviation_lines('green', mean, standard_deviation, standard_deviation_factor=2)

    plot_percentile_lines(feature_values, 70, color='darkblue')
    plot_percentile_lines(feature_values, 90, color='purple')
    plot_tukey_fence_upper_line(feature_values, 1.0, color='sienna')
    plot_tukey_fence_upper_line(feature_values, 1.5, color='teal')

    def annotate_outliers(outlier_values: pd.Series) -> None:
        """Annotate outlier bars with the corresponding short code unit name."""
        for rank, (item_index, value) in enumerate(outlier_values.items(), start=1):
            index = typing.cast(int, item_index)
            short_name = short_names[index]
            x_offset = (rank % 6) * 10
            if value > mean:
                x_offset = -x_offset
            plot.annotate(
                text=str(short_name),
                xy=(value, 1),
                xytext=(x_offset, 60),
                rotation=90,
                **plot_annotation_style,
            )

    annotate_outliers(feature_values.sort_values(ascending=False).head(number_of_outliers_to_annotate))
    annotate_outliers(feature_values.sort_values(ascending=True).head(number_of_outliers_to_annotate))

    plot.show()

## Normality Testing: Detailed Guide

### What is Normality Testing?

**Goal**: Decide whether your distribution is close enough to normal to safely use z-scores, or if you should use Tukey fence instead.

### The Three Tests & What They Tell You

#### 1. **Jarque-Bera Test** — "Is the shape normal?"
- **What it does**: Combines two checks (skewness + kurtosis)
- **Formula**: JB = (n/6) × (skewness² + (kurtosis/4)²)
- **Interpretation**:
  - Low JB value → p-value high → distribution looks normal
  - High JB value → p-value low → shape deviates from normal
- **Best for**: Moderate to large samples (n > 20)
- **Weakness**: Might miss subtle non-normality

#### 2. **Kolmogorov-Smirnov Test** — "How different is my CDF from normal's CDF?"
- **What it does**: Compares your actual distribution shape (CDF) to what a perfect normal distribution would look like
- **Visual idea**: Overlay two curves; measure max vertical distance
- **Interpretation**:
  - Small distance → distributions look similar → probably normal
  - Large distance → distributions look different → probably not normal
- **Best for**: All sample sizes; especially good at detecting shape changes
- **Strength**: Distribution-free (no assumptions about distribution)

#### 3. **Anderson-Darling Test** — "Are the tails weird? (Where outliers live)"
- **What it does**: Like K-S but gives extra weight to tail behavior
- **Why this matters**: Outliers live in the tails. Normal assumption breaks if tails are wrong.
- **Interpretation**:
  - Statistic < critical value (0.752) → probably normal
  - Statistic > critical value → probably not normal
- **Best for**: Detecting tail deviations (exactly what you care about!)
- **Strength**: Most sensitive to outlier behavior

### Quick Decision: When to Use What

| Situation | Recommendation |
|-----------|-----------------|
| **All 3 tests pass** + \|skewness\| < 0.5 | ✓ **Z-scores safe** |
| **2 tests pass** OR minimal skewness | ✓ **Either method; prefer Tukey** |
| **< 2 tests pass** OR \|skewness\| > 0.5 OR heavy tails | ✓ **Use Tukey fence** |

### Simpler Approach: Trust Your Eyes

Instead of (or in addition to) statistics, use the **Q-Q plot**:
- **Points on the line y=x?** → Normal ✓
- **S-shaped curve?** → Non-normal (heavy tails) → Use Tukey fence
- **Curvature at ends?** → Tail mismatch → Use Tukey fence


## Half-Normal Distributions: When Normality is "Folded"

### What is a Half-Normal Distribution?

A **half-normal distribution** occurs when:
- Your data contains **only positive values** (or only negative, or only above/below a threshold)
- The values **look like they were cut from a normal distribution at the mean**
- The shape resembles "half" of a normal bell curve

**Visual intuition**: Imagine a normal distribution. Now fold it at the center (mean = 0). What you see is a half-normal distribution.

### Where Does This Occur in Practice?

In code graph analysis, half-normal distributions appear with:

| Metric | Why Half-Normal | Example |
|--------|-----------------|---------|
| **PageRank** | Must be positive; rare to see negative centrality | Most packages have positive PageRank |
| **Degree** | Count of connections; can't be negative | Incoming + Outgoing dependencies are ≥ 0 |
| **Betweenness** | Percentage of paths; ranges 0-1 | Few nodes act as bridges; many have 0 |
| **Response Times** | Duration; can't be negative | Only positive time measurements |
| **Error Counts** | Frequency; can't be negative | Count-based metrics |

### Half-Normal vs. Normal: Key Difference

| Aspect | Normal Distribution | Half-Normal Distribution |
|--------|-------------------|------------------------|
| **Support** | All real numbers (-∞ to +∞) | Non-negative only (0 to +∞) |
| **Mean position** | Center of distribution | NOT at center; shifted right |
| **Skewness** | Can be symmetric or skewed | ALWAYS right-skewed (positive skew) |
| **Outlier detection** | Z-score works IF normal | Z-score fails; Tukey fence better |
| **Shape** | Bell curve | Half bell curve (one-sided) |

### Can We Test for Half-Normal?

**YES!** Here's the approach:

#### Step 1: Check if Distribution is "Folded"
- Mean is NOT at the center
- Minimum value near 0 or at a natural boundary
- Skewness strongly positive (> 0.5)

#### Step 2: Transform to Check Normality
- **Idea**: If it's truly half-normal, transform it back to normal
- **Method**: "Mirror" the data: create `transformed = -data`, concatenate with original
- **Result**: If the combined distribution becomes normal, original was half-normal

#### Step 3: Interpret Results
- Transformed distribution passes normality test → Original is **half-normal**
- Transformed distribution fails normality test → Original is **non-normal but not half-normal**

### Why This Matters for Outlier Detection

| Distribution Type | Recommended Method | Why |
|--|--|--|
| **Normal** | Z-score (|z| > 3) | Assumes bell curve symmetry |
| **Half-Normal** | Modified Z-score on original + Tukey | Tukey fence on positive half works |
| **Non-Normal** | Tukey fence (1.5 × IQR) | Robust to any shape |

**Key insight for half-normal**: 
- Standard Z-scores still work technically, BUT
- Tukey fence is safer because it doesn't assume symmetry
- Modified approach: Apply 1.5×IQR threshold only to the positive side



In [ ]:
def _compute_chi_square_cdf(x: float, df: int) -> float:
    """Approximate chi-square CDF using gamma function (scipy-free).
    
    BACKGROUND: Chi-square distribution is used in Jarque-Bera test.
    For df=2 (our case), formula is simply: 1 - exp(-x/2)
    
    This converts a test statistic into a p-value.
    Higher chi-square value → lower p-value → reject normality assumption.
    """
    if x <= 0:
        return 0.0
    if x > 100:
        return 1.0
    
    if df == 2:
        return float(1.0 - np.exp(-x / 2.0))
    
    if df == 1:
        return float(2.0 * (np.sqrt(x / np.pi) / 2.0 + np.arcsin(np.sqrt(x / (x + 1))) / np.pi))
    
    mean = float(df)
    var = float(2 * df)
    z = (x - mean) / np.sqrt(var)
    return float(0.5 * (1.0 + np.tanh(z / np.sqrt(2.0))))


def _compute_normal_cdf(z: typing.Union[float, np.ndarray]):
    """Compute standard normal CDF (cumulative distribution function).
    
    WHAT: For any value z (scalar or array), returns P(X ≤ z) where X ~ Normal(0,1).
    
    Example: CDF(0) = 0.5 (half the normal distribution is below mean)
             CDF(3) ≈ 0.9987 (almost everything is within ±3 standard deviations)
    
    Used in Kolmogorov-Smirnov and Anderson-Darling tests.
    Handles both scalar and array inputs through numpy broadcasting.
    """
    return 0.5 * (1.0 + np.tanh(z / np.sqrt(2.0)))


def compute_skewness(values: typing.Union[np.ndarray, typing.Sequence]) -> float:
    """Compute Fisher-Pearson coefficient of skewness.
    
    WHAT: Measures if distribution is symmetric or lopsided.
    
    Formula: skewness = E[(X - μ)³] / σ³
    
    Interpretation:
    ├─ Skewness ≈ 0 ────────────────► Symmetric (ideal for normal)
    ├─ Skewness ∈ (0, 0.5) ─────────► Slightly right-skewed (tail to right)
    ├─ Skewness > 0.5 ──────────────► Strongly right-skewed
    ├─ Skewness ∈ (-0.5, 0) ───────► Slightly left-skewed (tail to left)
    └─ Skewness < -0.5 ─────────────► Strongly left-skewed
    
    WHY: Skewed distributions have different outlier behavior.
    Normal tests expect symmetry, so skewness is a red flag.
    """
    arr = np.asarray(values)
    mean = np.mean(arr)
    std = np.std(arr, ddof=1)
    if std == 0:
        return 0.0
    m3 = np.mean((arr - mean) ** 3)
    return float(m3 / (std ** 3))


def compute_kurtosis(values: typing.Union[np.ndarray, typing.Sequence]) -> float:
    """Compute excess kurtosis (Fisher definition).
    
    WHAT: Measures how heavy or light the tails are compared to normal.
    
    Formula: excess_kurtosis = [E[(X - μ)⁴] / σ⁴] - 3
    (Subtract 3 to compare to normal distribution which has kurtosis = 3)
    
    Interpretation:
    ├─ Kurtosis ≈ 0 ───────────────► Normal tails (mesokurtic)
    ├─ Kurtosis > 1 ───────────────► Heavy tails (leptokurtic)
    │                               └─ Outliers more likely! Use Tukey fence
    ├─ Kurtosis < -1 ──────────────► Light tails (platykurtic)
    │                               └─ Fewer extreme values
    
    WHY: Heavy tails mean outliers are more probable. Z-scores assume normal tails,
    so high kurtosis breaks that assumption. Tukey fence is safer.
    """
    arr = np.asarray(values)
    mean = np.mean(arr)
    std = np.std(arr, ddof=1)
    if std == 0:
        return 0.0
    m4 = np.mean((arr - mean) ** 4)
    return float((m4 / (std ** 4)) - 3.0)


def test_for_half_normal(
    feature_values: pd.Series,
    feature_name: str,
    verbose: bool = True,
) -> dict[str, typing.Any]:
    """Test if a positive-valued distribution is half-normal.
    
    APPROACH:
    1. Check if data is bounded on one side (minimum ≈ 0 or natural boundary)
    2. Check for strong right skew (skewness > 0.5)
    3. Transform by mirroring: create inverted copy and concatenate
    4. Test if transformed distribution is normal
    5. If transformed is normal → original is half-normal
    
    A half-normal distribution is the result of taking the absolute value
    of a normal distribution, or equivalently, folding a normal distribution at zero.
    
    Args:
        feature_values: Series of positive values to test
        feature_name: Name for display
        verbose: Print results (True) or just return dictionary (False)
        
    Returns:
        Dictionary with:
        - 'is_half_normal': boolean (high confidence)
        - 'confidence': float 0-1 (0=definitely not, 1=definitely yes)
        - 'skewness': float (strong right skew is a sign)
        - 'minimum_value': float (should be near 0 or natural boundary)
        - 'transformation_test_passed': boolean (is transformed distribution normal?)
        - 'recommendation': string (what outlier method to use)
    """
    clean_values = feature_values.dropna().values
    n = len(clean_values)
    
    if n < 5:
        return {
            'is_half_normal': False,
            'confidence': 0.0,
            'reason': 'Insufficient data (n < 5)',
            'recommendation': 'Use Tukey fence (IQR-based)',
        }
    
    # Check 1: Is data non-negative or near-zero minimum?
    minimum_value = float(clean_values.min())
    maximum_value = float(clean_values.max())
    mean_value = float(clean_values.mean())
    
    if minimum_value < 0:
        return {
            'is_half_normal': False,
            'confidence': 0.0,
            'reason': 'Data contains negative values; cannot be half-normal',
            'recommendation': 'Use standard normality test on full distribution',
        }
    
    # Check 2: Is there strong right skew?
    skewness_value = compute_skewness(clean_values)
    kurtosis_value = compute_kurtosis(clean_values)
    
    # Half-normal should have skewness around 0.995 theoretically
    # But we will accept 0.3-2.0 as "half-normal-like"
    strong_right_skew = skewness_value > 0.3
    
    # Check 3: Is minimum very close to zero (folded at zero)?
    # Natural boundary for half-normal is at 0
    # Allow small tolerance for measurement error
    range_value = maximum_value - minimum_value
    minimum_relative_to_range = (minimum_value - 0) / range_value if range_value > 0 else 0
    bounded_at_zero = minimum_relative_to_range < 0.05  # Within 5% of lower bound
    
    # Check 4: Transform and test for normality
    # Mirror the data: if X ~ HalfNormal(σ), then {X, -X} ~ Normal(0, σ)
    transformed = np.concatenate([clean_values, -clean_values])
    
    # Test normality of transformed
    jarque_bera_stat_transformed = (len(transformed) / 6.0) * (
        compute_skewness(transformed) ** 2 + 
        (compute_kurtosis(transformed) / 4.0) ** 2
    )
    jarque_bera_p_transformed = 1.0 - _compute_chi_square_cdf(jarque_bera_stat_transformed, df=2)
    transformation_test_passed = jarque_bera_p_transformed > 0.05
    
    # Scoring confidence
    score = 0.0
    reasons = []
    
    if strong_right_skew:
        score += 0.3
        reasons.append(f"Strong right skew (skewness={skewness_value:+.3f})")
    
    if bounded_at_zero:
        score += 0.3
        reasons.append(f"Bounded near zero (minimum={minimum_value:.4f})")
    
    if transformation_test_passed:
        score += 0.4
        reasons.append(f"Mirrored distribution is normal (Jarque-Bera p={jarque_bera_p_transformed:.4f})")
    
    is_half_normal = score > 0.6
    
    if verbose:
        print(f"\n{'='*80}")
        print(f"HALF-NORMAL TEST: {feature_name}")
        print(f"{'='*80}")
        print(f"Sample size: {n}")
        print(f"\n📊 DISTRIBUTIONAL PROPERTIES")
        print(f"  Range: [{minimum_value:.4f}, {maximum_value:.4f}]")
        print(f"  Mean: {mean_value:.4f}")
        print(f"  Skewness: {skewness_value:+.4f} {'(✓ right-skewed)' if skewness_value > 0.3 else '(⚠ not skewed)'}")
        print(f"  Kurtosis: {kurtosis_value:+.4f}")
        
        print(f"\n🔍 HALF-NORMAL CHECKS")
        for reason in reasons:
            print(f"  ✓ {reason}")
        
        print(f"\n💡 CONCLUSION")
        print(f"  Confidence: {score:.1%}")
        print(f"  Classification: {'HALF-NORMAL ✓' if is_half_normal else 'NOT half-normal'}")
        
        if is_half_normal:
            print(f"\n🎯 RECOMMENDATION")
            print(f"  This distribution appears to be half-normal.")
            print(f"  └─ For outlier detection:")
            print(f"     • Tukey fence (1.5×IQR) is appropriate ✓ (robust to half-normal)")
            print(f"     • Z-scores work technically but less robust ⚠")
            print(f"     • Standard deviation assumes symmetry — may overestimate outliers")
        else:
            print(f"\n🎯 RECOMMENDATION")
            print(f"  Not clearly half-normal; use Tukey fence as default for positive-only data.")
        
        print(f"{'='*80}\n")
    
    return {
        'is_half_normal': is_half_normal,
        'confidence': score,
        'skewness': skewness_value,
        'kurtosis': kurtosis_value,
        'minimum_value': minimum_value,
        'maximum_value': maximum_value,
        'mean_value': mean_value,
        'transformation_test_passed': transformation_test_passed,
        'transformation_jarque_bera_p_value': jarque_bera_p_transformed,
        'reasons': reasons,
        'recommendation': 'Tukey fence (IQR-based, robust to half-normal distribution)',
    }


def quick_check_distribution_type(
    feature_values: pd.Series,
    feature_name: str,
) -> str:
    """Quick one-line classification: is it normal, half-normal, or other?
    
    Returns: "Normal", "Half-Normal", "Non-Normal", or "Insufficient Data"
    """
    clean_values = feature_values.dropna().values
    n = len(clean_values)
    
    if n < 3:
        return "Insufficient Data"
    
    # Check if positive-only
    if clean_values.min() >= 0 and clean_values.max() > 0:
        # Could be half-normal; check
        half_normal_result = test_for_half_normal(feature_values, feature_name, verbose=False)
        if half_normal_result['confidence'] > 0.6:
            return "Half-Normal"
    
    # Return non-normal for now; normality test function would go here
    return "Non-Normal"

In [ ]:
def distribution_report(
    feature_values: pd.Series,
    feature_name: str,
) -> None:
    """
    COMPREHENSIVE REPORT: Full details + clear recommendation.
    
    Prints:
    1. Summary statistics (mean, std, skewness, kurtosis)
    2. All three test results
    3. Direct recommendation + why
    """
    clean_values = feature_values.dropna().values
    n = len(clean_values)
    
    if n < 3:
        print(f"\n{feature_name}: Insufficient data (n={n}). Cannot test normality.")
        return
    
    print(f"\n{'='*80}")
    print(f"DISTRIBUTION REPORT: {feature_name}")
    print(f"{'='*80}")
    print(f"Sample size: {n}")
    
    mean = np.mean(clean_values)
    std = np.std(clean_values, ddof=1)
    skew = compute_skewness(clean_values)
    kurt = compute_kurtosis(clean_values)
    
    print(f"\n📊 DESCRIPTIVE STATISTICS")
    print(f"  Mean: {mean:.4f}")
    print(f"  Standard Deviation: {std:.4f}")
    print(f"  Skewness: {skew:+.4f} {'(✓ symmetric)' if abs(skew) < 0.3 else '(⚠ asymmetric)'}")
    print(f"  Kurtosis: {kurt:+.4f} {'(✓ normal tails)' if abs(kurt) < 0.5 else '(⚠ abnormal tails)'}")
    
    # Quick half-normal check for positive-only data
    if clean_values.min() >= 0 and clean_values.max() > 0:
        half_normal_check = test_for_half_normal(feature_values, feature_name, verbose=False)
        if half_normal_check['confidence'] > 0.6:
            print(f"\n⚠️  HALF-NORMAL DISTRIBUTION DETECTED")
            print(f"  Confidence: {half_normal_check['confidence']:.0%}")
            print(f"  → This appears to be a folded/truncated normal distribution")
            print(f"  → Tukey fence remains the best choice for outlier detection")
        else:
            print(f"  → Half-normal distribution not strongly indicated (confidence={half_normal_check['confidence']:.0%})")
    else:
        print(f"  → Data is not positive-only; half-normal check skipped")
    
    # Run the three tests
    sorted_vals = np.sort(clean_values)
    empirical_cdf = np.arange(1, n + 1) / n
    theoretical_cdf = _compute_normal_cdf((sorted_vals - mean) / std)
    
    jb_stat = (n / 6.0) * (skew ** 2 + (kurt / 4.0) ** 2)
    jb_p = 1.0 - _compute_chi_square_cdf(jb_stat, df=2)
    
    ks_stat = float(np.abs(empirical_cdf - theoretical_cdf).max())
    ks_p = float(2.0 * np.exp(-2.0 * (n * ks_stat) ** 2))
    ks_p = min(1.0, max(0.0, ks_p))
    
    anderson_stat = -n - (1.0 / n) * np.sum(
        np.log(theoretical_cdf + 1e-10) + np.log(1.0 - theoretical_cdf + 1e-10)
    )
    anderson_pass = anderson_stat < 0.752
    
    tests_pass = sum([jb_p > 0.05, ks_p > 0.05, anderson_pass])
    
    print(f"\n🧪 STATISTICAL TESTS (α=0.05)")
    print(f"  1. Jarque-Bera:       p={jb_p:.4f}  {'✓ PASS' if jb_p > 0.05 else '✗ FAIL'}")
    print(f"  2. Kolmogorov-Smirnov: p={ks_p:.4f}  {'✓ PASS' if ks_p > 0.05 else '✗ FAIL'}")
    print(f"  3. Anderson-Darling:  stat={anderson_stat:.3f}  {'✓ PASS' if anderson_pass else '✗ FAIL'}")
    print(f"\n  Result: {tests_pass}/3 tests passing")
    
    # Recommendation logic
    print(f"\n💡 RECOMMENDATION")
    if tests_pass >= 2 and abs(skew) < 0.5 and abs(kurt) < 1.0:
        print(f"  🎯 Use Z-scores (distribution is approximately normal)")
        print(f"     └─ Outlier threshold: |z| > 3")
    elif jb_p > 0.05 or (abs(skew) < 0.3 and abs(kurt) < 0.3):
        print(f"  🎯 Use either Z-scores or Tukey fence (minor deviations only)")
        print(f"     └─ Prefer Tukey fence for robustness")
        print(f"     └─ Tukey threshold: |value - median| > 1.5×IQR")
    else:
        print(f"  🎯 Use Tukey fence (distribution deviates significantly)")
        if abs(skew) > 0.5:
            print(f"     └─ Reason: Distribution is skewed {'right' if skew > 0 else 'left'}")
        if abs(kurt) > 1.0:
            print(f"     └─ Reason: Heavy tails (outliers more likely)")
        print(f"     └─ Tukey threshold: |value - median| > 1.5×IQR")
    
    print(f"{'='*80}\n")




In [ ]:
def plot_archetype_scatter(
    data: pd.DataFrame,
    x_column: str,
    y_column: str,
    archetype_column: str,
    archetype_color: str,
    archetype_label: str,
    title: str,
    x_label: str,
    y_label: str,
    top_n_annotations: int = 10,
    annotate_highest_y: bool = True,
) -> None:
    """Plot a 2D scatter to explore one archetype dimension.

    Highlights labeled archetype nodes with a distinct color and annotates the top_n_annotations
    most extreme points per axis. Each point is annotated at most once. Points appearing in both
    top-N lists show both axis values. Annotation format: axis#rank: name (axis=value).

    Args:
        data: Feature DataFrame as returned by query_archetype_data.
        x_column: Column name for the x-axis.
        y_column: Column name for the y-axis.
        archetype_column: Boolean column indicating archetype label membership.
        archetype_color: Color for labeled archetype nodes.
        archetype_label: Legend label for the archetype color.
        title: Plot title.
        x_label: x-axis label.
        y_label: y-axis label.
        top_n_annotations: Number of extreme points to annotate per axis.
        annotate_highest_y: If True, annotate the highest y values; if False, the lowest.
    """
    if data.empty:
        print("No data available to plot.")
        return

    unlabeled = data[~data[archetype_column]]
    labeled = data[data[archetype_column]]

    figure = plot.figure(figsize=(12, 8))
    plot.scatter(
        x=unlabeled[x_column],
        y=unlabeled[y_column],
        alpha=0.4,
        s=20,
        color='lightsteelblue',
        label='Unlabeled',
    )
    if not labeled.empty:
        plot.scatter(
            x=labeled[x_column],
            y=labeled[y_column],
            alpha=0.9,
            s=80,
            color=archetype_color,
            label=archetype_label,
            zorder=5,
        )

    plot.title(title, pad=20)
    plot.xlabel(x_label)
    plot.ylabel(y_label)
    plot.grid(True)

    # Tukey fence lines — vertical for x (upper), horizontal for y (upper or lower depending on archetype)
    x_upper_fence = compute_tukey_fence(data[x_column], fence_multiplier=1.5, upper=True)
    plot.axvline(x_upper_fence, color='teal', linestyle='dashdot', linewidth=1, alpha=0.7,
                 label=f"x Tukey upper fence (1.5xIQR): {x_upper_fence:.3f}")
    y_fence = compute_tukey_fence(data[y_column], fence_multiplier=1.5, upper=annotate_highest_y)
    fence_label = (f"y Tukey upper fence (Q3+1.5xIQR): {y_fence:.3f}" if annotate_highest_y
                   else f"y Tukey lower fence (Q1-1.5xIQR): {y_fence:.3f}")
    plot.axhline(y_fence, color='teal', linestyle='dashdot', linewidth=1, alpha=0.7, label=fence_label)
    plot.legend(fontsize=7)

    # Build per-axis rank maps (DataFrame index → rank within that axis top-N)
    top_x_rows = data.nlargest(top_n_annotations, x_column)
    top_y_rows = (
        data.nsmallest(top_n_annotations, y_column) if not annotate_highest_y
        else data.nlargest(top_n_annotations, y_column)
    )
    x_rank_by_index: dict[int, int] = {
        int(idx): rank for rank, idx in enumerate(top_x_rows.index, start=1)
    }
    y_rank_by_index: dict[int, int] = {
        int(idx): rank for rank, idx in enumerate(top_y_rows.index, start=1)
    }

    # Annotate each unique extreme point exactly once
    for position, data_index in enumerate(sorted(x_rank_by_index.keys() | y_rank_by_index.keys())):
        row = data.loc[data_index]
        y_offset = (position % 5) * 10

        x_rank = x_rank_by_index.get(data_index)
        y_rank = y_rank_by_index.get(data_index)

        rank_parts = []
        value_parts = []
        if x_rank is not None:
            rank_parts.append(f"x#{x_rank}")
            value_parts.append(f"x={row[x_column]:.3f}")
        if y_rank is not None:
            rank_parts.append(f"y#{y_rank}")
            value_parts.append(f"y={row[y_column]:.3f}")

        # Offset right for x-nominated points, left for y-only points
        text_offset_x = 5 if x_rank is not None else -80
        plot.annotate(
            text=f"{'/'.join(rank_parts)}: {truncate(str(row['shortCodeUnitName']))} ({', '.join(value_parts)})",
            xy=(row[x_column], row[y_column]),
            xytext=(text_offset_x, 5 + y_offset),
            **plot_annotation_style,
        )

    # Annotation key at the bottom of the figure
    annotation_key = (
        f"Annotation key — x#rank: top {top_n_annotations} by {x_label};  "
        f"y#rank: top {top_n_annotations} by {y_label}"
    )
    figure.text(0.5, 0.01, annotation_key, ha='center', va='bottom', fontsize=7, color='gray', style='italic')

    plot.tight_layout(rect=[0, 0.04, 1, 1])
    plot.show()

In [ ]:
def add_rank_column(
    data: pd.DataFrame,
    value_column_name: str,
    ascending: bool = False,
) -> pd.DataFrame:
    """Add a dense rank column to data. Rank 1 is the highest value when ascending=False.

    The rank column is named value_column_name + '_ranking'. Does nothing if it already exists.
    """
    if value_column_name not in data.columns:
        raise ValueError(f"Column '{value_column_name}' does not exist in the DataFrame.")
    if data.empty:
        return data
    rank_column_name = value_column_name + '_ranking'
    if rank_column_name in data.columns:
        return data
    data[rank_column_name] = data[value_column_name].rank(ascending=ascending, method='dense').astype(int)
    return data


def pareto_frontier(
    input_data: pd.DataFrame,
    metrics: list[str],
    maximize: bool = True,
) -> pd.DataFrame:
    """Return the Pareto-optimal (non-dominated) rows from input_data.

    With maximize=False the frontier is computed in the direction of maximizing all metrics,
    meaning nodes that are not dominated by any other node when trying to maximize all metrics
    simultaneously. This is useful for finding the most extreme nodes.

    Args:
        input_data: Source DataFrame.
        metrics: Column names to evaluate.
        maximize: When False (default for finding most extreme nodes), identifies rows where
                  no other row has strictly higher values across all metrics.

    Returns:
        DataFrame with only non-dominated rows, in the original row order.
    """
    values = input_data[metrics].to_numpy(dtype=float)
    if not maximize:
        values = -values
    is_not_on_frontier = np.zeros(len(values), dtype=bool)
    for i, point in enumerate(values):
        if is_not_on_frontier[i]:
            continue
        dominates_point = np.all(values >= point, axis=1) & np.any(values > point, axis=1)
        is_not_on_frontier |= dominates_point
    return input_data[~is_not_on_frontier].reset_index(drop=True)


def get_best_archetype_trade_offs(
    data: pd.DataFrame,
    feature_names: list[str],
    code_unit_columns: list[str] | None = None,
    top_n: int = 10,
) -> pd.DataFrame:
    """Return the top_n code units on the Pareto frontier for the given feature_names.

    Finds nodes that represent the best simultaneous trade-offs: no other node has higher
    values for all of the specified features at once. Sorts by the sum of feature ranks
    so the overall most extreme nodes appear first.

    Args:
        data: Feature DataFrame with pre-computed rank columns (add_rank_column).
        feature_names: Raw feature columns to evaluate. Higher values are treated as more extreme.
        code_unit_columns: Identifier columns to include in the result.
        top_n: Maximum number of rows to return.

    Returns:
        Filtered and sorted DataFrame with identifier columns, feature values, and rankings.
    """
    if code_unit_columns is None:
        code_unit_columns = ['projectName', 'shortCodeUnitName', 'codeUnitName']
    if data.empty:
        print("DataFrame is empty. No Pareto frontier can be computed.")
        return data

    ranking_columns = [feature + '_ranking' for feature in feature_names]
    available_ranking_columns = [column for column in ranking_columns if column in data.columns]

    pareto_best = pareto_frontier(data, feature_names, maximize=False)

    if available_ranking_columns:
        pareto_best = pareto_best.copy()
        pareto_best['_combined_rank_score'] = pareto_best[available_ranking_columns].sum(axis=1)
        pareto_best = pareto_best.sort_values('_combined_rank_score').head(top_n)
        pareto_best = pareto_best.drop(columns=['_combined_rank_score'])
    else:
        pareto_best = pareto_best.head(top_n)

    selected_columns = [
        column for column in code_unit_columns + feature_names + available_ranking_columns
        if column in pareto_best.columns
    ]
    return pareto_best[selected_columns]

## 1. Java Packages

In [ ]:
java_package_features = query_archetype_data("Package")
display(java_package_features.head(5))

### 1.1 PageRank Distribution

PageRank measures how important a node is based on the number and quality of incoming dependencies.
A right-skewed distribution is typical: most packages have low PageRank, while a few are widely depended on.
Outliers on the high end are candidates for the Authority archetype.

In [ ]:
distribution_report(java_package_features['pageRank'], 'PageRank')

In [ ]:
plot_feature_distribution(
    java_package_features['pageRank'],
    feature_name='PageRank',
    short_names=java_package_features['shortCodeUnitName'],
    title_prefix='Java Package',
    log_y_axis=True,
)

### 1.2 ArticleRank Distribution

ArticleRank places more weight on links from nodes with fewer outgoing connections.
Comparing the ArticleRank distribution to PageRank reveals structural asymmetries.
A node with unusually high PageRank but low ArticleRank is a strong Authority candidate.

In [ ]:
distribution_report(java_package_features['articleRank'], 'ArticleRank')

In [ ]:
plot_feature_distribution(
    java_package_features['articleRank'],
    feature_name='ArticleRank',
    short_names=java_package_features['shortCodeUnitName'],
    title_prefix='Java Package',
    log_y_axis=True,
)

### 1.3 PageRank-to-ArticleRank Difference Distribution

This is the normalized difference: `normalizedPageRank - normalizedArticleRank`.
A positive value means the node is depended on proportionally more than it depends on others.
High positive values are the defining signal for the Authority archetype.

In [ ]:
distribution_report(java_package_features['pageToArticleRankDifference'], 'PageRank-to-ArticleRank Difference (normalized)')

In [ ]:
plot_feature_distribution(
    java_package_features['pageToArticleRankDifference'],
    feature_name='PageRank-to-ArticleRank Difference (normalized)',
    short_names=java_package_features['shortCodeUnitName'],
    title_prefix='Java Package',
)

### 1.4 Betweenness Distribution

Betweenness centrality counts how often a node lies on the shortest path between two other nodes.
Most code units will have zero or near-zero betweenness. A small number of packages that bridge
subsystems will stand out as outliers — these are candidates for the Bottleneck archetype.

In [ ]:
distribution_report(java_package_features['betweenness'], 'Betweenness')

In [ ]:
plot_feature_distribution(
    java_package_features['betweenness'],
    feature_name='Betweenness',
    short_names=java_package_features['shortCodeUnitName'],
    title_prefix='Java Package',
    log_y_axis=True,
)

### 1.5 Local Clustering Coefficient Distribution

The local clustering coefficient measures how well a node's direct neighbors are connected to each other.
A value of 1.0 means all neighbors are also connected; a value of 0.0 means no neighbors are connected.
Hub candidates have high degree combined with a low local clustering coefficient.

In [ ]:
distribution_report(java_package_features['localClusteringCoefficient'], 'Local Clustering Coefficient')

In [ ]:
plot_feature_distribution(
    java_package_features['localClusteringCoefficient'],
    feature_name='Local Clustering Coefficient',
    short_names=java_package_features['shortCodeUnitName'],
    title_prefix='Java Package',
    standard_deviation_lines=False,
)

### 1.6 Degree Distribution

Degree is the total number of dependency edges (incoming + outgoing).
A heavy right tail indicates a small number of highly connected packages — classic power-law behaviour.
Combined with a low local clustering coefficient, high degree is the defining signal for the Hub archetype.

In [ ]:
distribution_report(java_package_features['degree'], 'Degree')

In [ ]:
plot_feature_distribution(
    java_package_features['degree'],
    feature_name='Degree',
    short_names=java_package_features['shortCodeUnitName'],
    title_prefix='Java Package',
    log_y_axis=True,
)

### 1.7 Incoming Dependencies Distribution

In [ ]:
distribution_report(java_package_features['incomingDependencies'], 'Incoming Dependencies')

In [ ]:
plot_feature_distribution(
    java_package_features['incomingDependencies'],
    feature_name='Incoming Dependencies',
    short_names=java_package_features['shortCodeUnitName'],
    title_prefix='Java Package',
    log_y_axis=True,
)

### 1.8 Outgoing Dependencies Distribution

In [ ]:
distribution_report(java_package_features['outgoingDependencies'], 'Outgoing Dependencies')

In [ ]:
plot_feature_distribution(
    java_package_features['outgoingDependencies'],
    feature_name='Outgoing Dependencies',
    short_names=java_package_features['shortCodeUnitName'],
    title_prefix='Java Package',
    log_y_axis=True,
)

### 1.9 Authority: PageRank vs PageRank-to-ArticleRank Difference

The Authority archetype lies in the **top-right corner**: high PageRank (widely referenced) combined with a
large positive difference between normalized PageRank and normalized ArticleRank (referenced proportionally more than it references others).

Orange points have been labeled as Authority archetypes by the query. The top 10 outliers per axis are annotated.

In [ ]:
plot_archetype_scatter(
    data=java_package_features,
    x_column='pageRank',
    y_column='pageToArticleRankDifference',
    archetype_column='isAuthorityArchetype',
    archetype_color='darkorange',
    archetype_label='Authority archetype',
    title='Java Package Authority: PageRank vs PageRank-to-ArticleRank Difference',
    x_label='PageRank',
    y_label='PageRank-to-ArticleRank Difference (normalized)',
    annotate_highest_y=True,
)

### 1.10 Hub: Degree vs Local Clustering Coefficient

The Hub archetype lies in the **bottom-right corner**: high degree (many connections) combined with a
low local clustering coefficient (neighbors are not well connected to each other).

Red points have been labeled as Hub archetypes by the query. The top 10 outliers per axis are annotated:
the highest degree nodes on the right, and the lowest clustering coefficient nodes on the bottom.

In [ ]:
plot_archetype_scatter(
    data=java_package_features,
    x_column='degree',
    y_column='localClusteringCoefficient',
    archetype_column='isHubArchetype',
    archetype_color='crimson',
    archetype_label='Hub archetype',
    title='Java Package Hub: Degree vs Local Clustering Coefficient',
    x_label='Degree (incoming + outgoing dependencies)',
    y_label='Local Clustering Coefficient',
    annotate_highest_y=False,  # annotate lowest clustering coefficient — most Hub-like
)

In [ ]:
ARCHETYPE_FEATURES_TO_RANK = [
    'degree',
    'pageRank',
    'articleRank',
    'pageToArticleRankDifference',
    'betweenness',
    'inverseLocalClusteringCoefficient',
    'incomingDependencies',
    'outgoingDependencies',
]

for feature in ARCHETYPE_FEATURES_TO_RANK:
    java_package_features = add_rank_column(java_package_features, feature, ascending=False)

### 1.11 Pareto Best Trade-offs

Multi-objective Pareto analysis finds code units where no other unit has a simultaneously higher value
for all selected features. These are the best trade-offs: to improve one feature score you would have
to accept a lower score on at least one other feature.

Results are sorted by the sum of feature ranks so the most overall extreme units appear first.

#### 1.11.1 All Archetype Features

Code units extreme across all archetype-relevant dimensions simultaneously.

In [ ]:
display(get_best_archetype_trade_offs(java_package_features, ARCHETYPE_FEATURES_TO_RANK))

#### 1.11.2 Hub (High Degree, Low Local Clustering Coefficient)

A Hub has many direct connections whose neighbors are not connected to each other.
High degree combined with high inverse clustering coefficient (= low clustering coefficient)
identifies packages that are used everywhere but do not belong to any tight module cluster.
Potential God classes or over-used utilities that violate modular design.

In [ ]:
hub_features = ['degree', 'inverseLocalClusteringCoefficient']
display(get_best_archetype_trade_offs(java_package_features, hub_features))

#### 1.11.3 Bottleneck (High Betweenness)

A Bottleneck lies on many shortest dependency paths between other nodes.
High betweenness signals an architectural chokepoint: removing or changing this package
would disrupt communication between multiple subsystems.
Often an unintended dependency concentration that accumulates over time.

In [ ]:
bottleneck_features = ['betweenness', 'degree']
display(get_best_archetype_trade_offs(java_package_features, bottleneck_features))

#### 1.11.4 Authority (High PageRank, High PageRank-to-ArticleRank Difference)

An Authority is widely referenced (high PageRank) but does not reference others proportionally
(high normalized difference between PageRank and ArticleRank).
Typical examples are core utility libraries, base classes, or framework entry points that
many modules depend on without the relationship being reciprocal.

In [ ]:
authority_features = ['pageRank', 'pageToArticleRankDifference']
display(get_best_archetype_trade_offs(java_package_features, authority_features))

## 2. Java Types

In [ ]:
java_type_features = query_archetype_data("Type")
display(java_type_features.head(5))

### 2.1 PageRank Distribution

In [ ]:
distribution_report(java_type_features['pageRank'], 'PageRank')

In [ ]:
plot_feature_distribution(
    java_type_features['pageRank'],
    feature_name='PageRank',
    short_names=java_type_features['shortCodeUnitName'],
    title_prefix='Java Type',
    log_y_axis=True,
)

### 2.2 ArticleRank Distribution

In [ ]:
distribution_report(java_type_features['articleRank'], 'ArticleRank')

In [ ]:
plot_feature_distribution(
    java_type_features['articleRank'],
    feature_name='ArticleRank',
    short_names=java_type_features['shortCodeUnitName'],
    title_prefix='Java Type',
    log_y_axis=True,
)

### 2.3 PageRank-to-ArticleRank Difference Distribution

In [ ]:
distribution_report(java_type_features['pageToArticleRankDifference'], 'PageRank-to-ArticleRank Difference (normalized)')

In [ ]:
plot_feature_distribution(
    java_type_features['pageToArticleRankDifference'],
    feature_name='PageRank-to-ArticleRank Difference (normalized)',
    short_names=java_type_features['shortCodeUnitName'],
    title_prefix='Java Type',
)

### 2.4 Betweenness Distribution

In [ ]:
distribution_report(java_type_features['betweenness'], 'Betweenness')

In [ ]:
plot_feature_distribution(
    java_type_features['betweenness'],
    feature_name='Betweenness',
    short_names=java_type_features['shortCodeUnitName'],
    title_prefix='Java Type',
    log_y_axis=True,
)

### 2.5 Local Clustering Coefficient Distribution

In [ ]:
distribution_report(java_type_features['localClusteringCoefficient'], 'Local Clustering Coefficient')

In [ ]:
plot_feature_distribution(
    java_type_features['localClusteringCoefficient'],
    feature_name='Local Clustering Coefficient',
    short_names=java_type_features['shortCodeUnitName'],
    title_prefix='Java Type',
    standard_deviation_lines=False,
)

### 2.6 Degree Distribution

In [ ]:
distribution_report(java_type_features['degree'], 'Degree')

In [ ]:
plot_feature_distribution(
    java_type_features['degree'],
    feature_name='Degree',
    short_names=java_type_features['shortCodeUnitName'],
    title_prefix='Java Type',
    log_y_axis=True,
)

### 2.7 Incoming Dependencies Distribution

In [ ]:
distribution_report(java_type_features['incomingDependencies'], 'Incoming Dependencies')

In [ ]:
plot_feature_distribution(
    java_type_features['incomingDependencies'],
    feature_name='Incoming Dependencies',
    short_names=java_type_features['shortCodeUnitName'],
    title_prefix='Java Type',
    log_y_axis=True,
)

### 2.8 Outgoing Dependencies Distribution

In [ ]:
distribution_report(java_type_features['outgoingDependencies'], 'Outgoing Dependencies')

In [ ]:
plot_feature_distribution(
    java_type_features['outgoingDependencies'],
    feature_name='Outgoing Dependencies',
    short_names=java_type_features['shortCodeUnitName'],
    title_prefix='Java Type',
    log_y_axis=True,
)

### 2.9 Authority: PageRank vs PageRank-to-ArticleRank Difference

In [ ]:
plot_archetype_scatter(
    data=java_type_features,
    x_column='pageRank',
    y_column='pageToArticleRankDifference',
    archetype_column='isAuthorityArchetype',
    archetype_color='darkorange',
    archetype_label='Authority archetype',
    title='Java Type Authority: PageRank vs PageRank-to-ArticleRank Difference',
    x_label='PageRank',
    y_label='PageRank-to-ArticleRank Difference (normalized)',
    annotate_highest_y=True,
)

### 2.10 Hub: Degree vs Local Clustering Coefficient

In [ ]:
plot_archetype_scatter(
    data=java_type_features,
    x_column='degree',
    y_column='localClusteringCoefficient',
    archetype_column='isHubArchetype',
    archetype_color='crimson',
    archetype_label='Hub archetype',
    title='Java Type Hub: Degree vs Local Clustering Coefficient',
    x_label='Degree (incoming + outgoing dependencies)',
    y_label='Local Clustering Coefficient',
    annotate_highest_y=False,
)

In [ ]:
for feature in ARCHETYPE_FEATURES_TO_RANK:
    java_type_features = add_rank_column(java_type_features, feature, ascending=False)

### 2.11 Pareto Best Trade-offs

#### 2.11.1 All Archetype Features

In [ ]:
display(get_best_archetype_trade_offs(java_type_features, ARCHETYPE_FEATURES_TO_RANK))

#### 2.11.2 Hub (High Degree, Low Local Clustering Coefficient)

In [ ]:
display(get_best_archetype_trade_offs(java_type_features, hub_features))

#### 2.11.3 Bottleneck (High Betweenness)

In [ ]:
display(get_best_archetype_trade_offs(java_type_features, bottleneck_features))

#### 2.11.4 Authority (High PageRank, High PageRank-to-ArticleRank Difference)

In [ ]:
display(get_best_archetype_trade_offs(java_type_features, authority_features))

## 3. SCIP Modules

SCIP (Semantic Code Intelligence Protocol) modules map to source directories.
This section covers multi-language codebases indexed via SCIP — Go, TypeScript, Rust, Java, and others.
The same archetype metrics apply: PageRank reflects how widely a module is imported; betweenness identifies bridge directories; local clustering coefficient and degree identify Hub modules.

In [ ]:
scip_module_features = query_archetype_data("SemanticCodeIndexModule")
display(scip_module_features.head(5))

### 3.1 PageRank Distribution

In [ ]:
distribution_report(scip_module_features['pageRank'], 'PageRank')

In [ ]:
plot_feature_distribution(
    scip_module_features['pageRank'],
    feature_name='PageRank',
    short_names=scip_module_features['shortCodeUnitName'],
    title_prefix='SCIP Module',
    log_y_axis=True,
)

### 3.2 ArticleRank Distribution

In [ ]:
distribution_report(scip_module_features['articleRank'], 'ArticleRank')

In [ ]:
plot_feature_distribution(
    scip_module_features['articleRank'],
    feature_name='ArticleRank',
    short_names=scip_module_features['shortCodeUnitName'],
    title_prefix='SCIP Module',
    log_y_axis=True,
)

### 3.3 PageRank-to-ArticleRank Difference Distribution

In [ ]:
distribution_report(scip_module_features['pageToArticleRankDifference'], 'PageRank-to-ArticleRank Difference (normalized)')

In [ ]:
plot_feature_distribution(
    scip_module_features['pageToArticleRankDifference'],
    feature_name='PageRank-to-ArticleRank Difference (normalized)',
    short_names=scip_module_features['shortCodeUnitName'],
    title_prefix='SCIP Module',
)

### 3.4 Betweenness Distribution

In [ ]:
distribution_report(scip_module_features['betweenness'], 'Betweenness')

In [ ]:
plot_feature_distribution(
    scip_module_features['betweenness'],
    feature_name='Betweenness',
    short_names=scip_module_features['shortCodeUnitName'],
    title_prefix='SCIP Module',
    log_y_axis=True,
)

### 3.5 Local Clustering Coefficient Distribution

In [ ]:
distribution_report(scip_module_features['localClusteringCoefficient'], 'Local Clustering Coefficient')

In [ ]:
plot_feature_distribution(
    scip_module_features['localClusteringCoefficient'],
    feature_name='Local Clustering Coefficient',
    short_names=scip_module_features['shortCodeUnitName'],
    title_prefix='SCIP Module',
    standard_deviation_lines=False,
)

### 3.6 Degree Distribution

In [ ]:
distribution_report(scip_module_features['degree'], 'Degree')

In [ ]:
plot_feature_distribution(
    scip_module_features['degree'],
    feature_name='Degree',
    short_names=scip_module_features['shortCodeUnitName'],
    title_prefix='SCIP Module',
    log_y_axis=True,
)

### 3.7 Incoming Dependencies Distribution

In [ ]:
distribution_report(scip_module_features['incomingDependencies'], 'Incoming Dependencies')

In [ ]:
plot_feature_distribution(
    scip_module_features['incomingDependencies'],
    feature_name='Incoming Dependencies',
    short_names=scip_module_features['shortCodeUnitName'],
    title_prefix='SCIP Module',
    log_y_axis=True,
)

### 3.8 Outgoing Dependencies Distribution

In [ ]:
distribution_report(scip_module_features['outgoingDependencies'], 'Outgoing Dependencies')

In [ ]:
plot_feature_distribution(
    scip_module_features['outgoingDependencies'],
    feature_name='Outgoing Dependencies',
    short_names=scip_module_features['shortCodeUnitName'],
    title_prefix='SCIP Module',
    log_y_axis=True,
)

### 3.9 Authority: PageRank vs PageRank-to-ArticleRank Difference

In [ ]:
plot_archetype_scatter(
    data=scip_module_features,
    x_column='pageRank',
    y_column='pageToArticleRankDifference',
    archetype_column='isAuthorityArchetype',
    archetype_color='darkorange',
    archetype_label='Authority archetype',
    title='SCIP Module Authority: PageRank vs PageRank-to-ArticleRank Difference',
    x_label='PageRank',
    y_label='PageRank-to-ArticleRank Difference (normalized)',
    annotate_highest_y=True,
)

### 3.10 Hub: Degree vs Local Clustering Coefficient

In [ ]:
plot_archetype_scatter(
    data=scip_module_features,
    x_column='degree',
    y_column='localClusteringCoefficient',
    archetype_column='isHubArchetype',
    archetype_color='crimson',
    archetype_label='Hub archetype',
    title='SCIP Module Hub: Degree vs Local Clustering Coefficient',
    x_label='Degree (incoming + outgoing dependencies)',
    y_label='Local Clustering Coefficient',
    annotate_highest_y=False,
)

In [ ]:
for feature in ARCHETYPE_FEATURES_TO_RANK:
    scip_module_features = add_rank_column(scip_module_features, feature, ascending=False)

### 3.11 Pareto Best Trade-offs

#### 3.11.1 All Archetype Features

In [ ]:
display(get_best_archetype_trade_offs(scip_module_features, ARCHETYPE_FEATURES_TO_RANK))

#### 3.11.2 Hub

In [ ]:
display(get_best_archetype_trade_offs(scip_module_features, hub_features))

#### 3.11.3 Bottleneck

In [ ]:
display(get_best_archetype_trade_offs(scip_module_features, bottleneck_features))

#### 3.11.4 Authority

In [ ]:
display(get_best_archetype_trade_offs(scip_module_features, authority_features))

## 4. SCIP Internal Types

SCIP internal types are the finest-grained unit — individual classes, interfaces, structs, or functions within the indexed codebase.
This section can surface type-level archetypes such as widely imported utility types (Authority) or types that cross many module boundaries (Bottleneck).
Note that the number of nodes may be large; queries require the archetype features to have been run first.

In [ ]:
scip_internal_type_features = query_archetype_data("SemanticCodeIndexInternalType")
display(scip_internal_type_features.head(5))

### 4.1 PageRank Distribution

In [ ]:
distribution_report(scip_internal_type_features['pageRank'], 'PageRank')

In [ ]:
plot_feature_distribution(
    scip_internal_type_features['pageRank'],
    feature_name='PageRank',
    short_names=scip_internal_type_features['shortCodeUnitName'],
    title_prefix='SCIP Internal Type',
    log_y_axis=True,
)

### 4.2 ArticleRank Distribution

In [ ]:
distribution_report(scip_internal_type_features['articleRank'], 'ArticleRank')

In [ ]:
plot_feature_distribution(
    scip_internal_type_features['articleRank'],
    feature_name='ArticleRank',
    short_names=scip_internal_type_features['shortCodeUnitName'],
    title_prefix='SCIP Internal Type',
    log_y_axis=True,
)

### 4.3 PageRank-to-ArticleRank Difference Distribution

In [ ]:
distribution_report(scip_internal_type_features['pageToArticleRankDifference'], 'PageRank-to-ArticleRank Difference (normalized)')

In [ ]:
plot_feature_distribution(
    scip_internal_type_features['pageToArticleRankDifference'],
    feature_name='PageRank-to-ArticleRank Difference (normalized)',
    short_names=scip_internal_type_features['shortCodeUnitName'],
    title_prefix='SCIP Internal Type',
)

### 4.4 Betweenness Distribution

In [ ]:
distribution_report(scip_internal_type_features['betweenness'], 'Betweenness')

In [ ]:
plot_feature_distribution(
    scip_internal_type_features['betweenness'],
    feature_name='Betweenness',
    short_names=scip_internal_type_features['shortCodeUnitName'],
    title_prefix='SCIP Internal Type',
    log_y_axis=True,
)

### 4.5 Local Clustering Coefficient Distribution

In [ ]:
distribution_report(scip_internal_type_features['localClusteringCoefficient'], 'Local Clustering Coefficient')

In [ ]:
plot_feature_distribution(
    scip_internal_type_features['localClusteringCoefficient'],
    feature_name='Local Clustering Coefficient',
    short_names=scip_internal_type_features['shortCodeUnitName'],
    title_prefix='SCIP Internal Type',
    standard_deviation_lines=False,
)

### 4.6 Degree Distribution

In [ ]:
distribution_report(scip_internal_type_features['degree'], 'Degree')

In [ ]:
plot_feature_distribution(
    scip_internal_type_features['degree'],
    feature_name='Degree',
    short_names=scip_internal_type_features['shortCodeUnitName'],
    title_prefix='SCIP Internal Type',
    log_y_axis=True,
)

### 4.7 Incoming Dependencies Distribution

In [ ]:
distribution_report(scip_internal_type_features['incomingDependencies'], 'Incoming Dependencies')

In [ ]:
plot_feature_distribution(
    scip_internal_type_features['incomingDependencies'],
    feature_name='Incoming Dependencies',
    short_names=scip_internal_type_features['shortCodeUnitName'],
    title_prefix='SCIP Internal Type',
    log_y_axis=True,
)

### 4.8 Outgoing Dependencies Distribution

In [ ]:
distribution_report(scip_internal_type_features['outgoingDependencies'], 'Outgoing Dependencies')

In [ ]:
plot_feature_distribution(
    scip_internal_type_features['outgoingDependencies'],
    feature_name='Outgoing Dependencies',
    short_names=scip_internal_type_features['shortCodeUnitName'],
    title_prefix='SCIP Internal Type',
    log_y_axis=True,
)

### 4.9 Authority: PageRank vs PageRank-to-ArticleRank Difference

In [ ]:
plot_archetype_scatter(
    data=scip_internal_type_features,
    x_column='pageRank',
    y_column='pageToArticleRankDifference',
    archetype_column='isAuthorityArchetype',
    archetype_color='darkorange',
    archetype_label='Authority archetype',
    title='SCIP Internal Type Authority: PageRank vs PageRank-to-ArticleRank Difference',
    x_label='PageRank',
    y_label='PageRank-to-ArticleRank Difference (normalized)',
    annotate_highest_y=True,
)

### 4.10 Hub: Degree vs Local Clustering Coefficient

In [ ]:
plot_archetype_scatter(
    data=scip_internal_type_features,
    x_column='degree',
    y_column='localClusteringCoefficient',
    archetype_column='isHubArchetype',
    archetype_color='crimson',
    archetype_label='Hub archetype',
    title='SCIP Internal Type Hub: Degree vs Local Clustering Coefficient',
    x_label='Degree (incoming + outgoing dependencies)',
    y_label='Local Clustering Coefficient',
    annotate_highest_y=False,
)

In [ ]:
for feature in ARCHETYPE_FEATURES_TO_RANK:
    scip_internal_type_features = add_rank_column(scip_internal_type_features, feature, ascending=False)

### 4.11 Pareto Best Trade-offs

#### 4.11.1 All Archetype Features

In [ ]:
display(get_best_archetype_trade_offs(scip_internal_type_features, ARCHETYPE_FEATURES_TO_RANK))

#### 4.11.2 Hub

In [ ]:
display(get_best_archetype_trade_offs(scip_internal_type_features, hub_features))

#### 4.11.3 Bottleneck

In [ ]:
display(get_best_archetype_trade_offs(scip_internal_type_features, bottleneck_features))

#### 4.11.4 Authority

In [ ]:
display(get_best_archetype_trade_offs(scip_internal_type_features, authority_features))